In [1]:
!java -version

java version "1.8.0_503"
Java(TM) SE Runtime Environment (build 1.8.0_503-b01)
Java HotSpot(TM) Client VM (build 25.503-b01, mixed mode, sharing)


In [60]:
import tabula
import numpy as np

tabula.environment_info()

Python version:
    3.14.7 | packaged by conda-forge | (main, Aug 31 2026, 07:22:37) [MSC v.1944 64 bit (AMD64)]
Java version:
    java version "1.8.0_503"
Java(TM) SE Runtime Environment (build 1.8.0_503-b01)
Java HotSpot(TM) Client VM (build 25.503-b01, mixed mode, sharing)
tabula-py version: 2.10.0
platform: Windows-11-10.0.26200-SP0
uname:
    uname_result(system='Windows', node='Morgans_Desktop', release='11', version='10.0.26200', machine='AMD64')
linux_distribution: ('', '', '')
mac_ver: ('', ('', '', ''), '')


In [3]:
pdf_path = "https://github.com/chezou/tabula-py/raw/master/tests/resources/data.pdf"

dfs = tabula.read_pdf(pdf_path, stream=True)
# read_pdf returns list of DataFrames
print(len(dfs))
dfs[0]

Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'
'pages' argument isn't specified.Will extract only from page 1 by default.


1


,Unnamed: 0,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2
5,Valiant,18.1,6,225.0,105,2.76,3.460,20.22,1,0,3,1
6,Duster 360,14.3,8,360.0,245,3.21,3.570,15.84,0,0,3,4
7,Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2
8,Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2
9,Merc 280,19.2,6,167.6,123,3.92,3.440,18.30,1,0,4,4


In [20]:
pdf_path = "C:\\Users\\willi\\Documents\\monthly_budget\\data\\raw\\tesco.pdf"

dfs = tabula.read_pdf(pdf_path, stream=True, pandas_options={'header':None}, pages='all')
# read_pdf returns list of DataFrames
print(len(dfs))
dfs

4


[                                                    0
 0          SUMMARY BOX — Cardholder statement version
 1   The information contained in this table summar...
 2                                         conditions.
 3   Interest Free Period • Maximum 56 days for pur...
 4   have paid the previous month’s balance in full...
 5   • There is no interest free period on cash w i...
 6   Interest Charging Information You w on’t pay i...
 7   • repay all balances (excluding any balances o...
 8   • have paid all balances (excluding any balanc...
 9   Otherw ise, the period over w hich interest is...
 10                                         From Until
 11  Purchases, Cash withdrawals, Cash From date de...
 12          transactions, Balance transfers and Money
 13                                          transfers
 14  If you don't pay your balance in full (excludi...
 15  charged on the average daily balance until ful...
 16  make a payment, the more interest w ill be cha...
 17  † lf 

In [71]:
for i, df in enumerate(dfs):
    if df.shape[1] > 1:
        print(f"DataFrame {i}:")
        df = df.dropna(axis=0, thresh=4)
        df = df[(df.iloc[:, 0].apply(_is_date_string) & df.iloc[:, 1].apply(_is_date_string))]
        df.iloc[:, 3] = df.iloc[:, 3:-1].replace(np.nan, '').astype('str').agg('-'.join, axis=1)
        df = df.iloc[:, [0, 1, 2, 3, -1]]

        
        df.columns = ['trans_date', 'post_date', 'id', 'description', 'amount']
        df['amount'] = df['amount'].str.replace('£', '').str.replace(',', '').astype(float)
        df = df.dropna(subset=['id'])
        df['id'] = df['id'].astype(int)
        print(df.head())


DataFrame 2:
  trans_date post_date        id                         description  amount
0     25 JUL    27 JUL  30022364  LYMEFIELD GARDEN CEN L HYDE ENG---    8.80
1     26 JUL    27 JUL  19719802   WWW.BLOOMANDWILD.COM LONDON-LND--   20.54
2     26 JUL    27 JUL  10248687   WWW.ZIPWORLD.CO.UK LLANRWST-WLS--   15.75
3     26 JUL    27 JUL  35394067        NATIONAL TRUST ALTRINCHAM---   15.30
4     26 JUL    27 JUL  32921599        NATIONAL TRUST ALTRINCHAM---    3.00
DataFrame 3:
  trans_date post_date        id                        description  amount
4     14 AUG    17 AUG  97639116  Tameside MBC Parking TAMESIDE TAM    1.00
5     14 AUG    17 AUG   2017786               ALDI STALYBRIDGE ENG   11.77
6     14 AUG    17 AUG  24547209   SumUp *LITTLE TOWN S Stalybridge    7.95
7     14 AUG    17 AUG  15567337    Zettle_*Tigellae Limit Hyde GBR   15.50
8     15 AUG    17 AUG  63146633                 PONY PATCH GLOSSOP   26.00


In [68]:
df.amount

0       £8.80
1      £20.54
2      £15.75
3      £15.30
4       £3.00
5      £69.08
6       £6.20
7       £2.50
8     £101.84
9      £27.35
10      £9.95
11     £22.40
12     £14.50
13     £12.64
14      £6.28
15     £18.35
16     £30.00
17      £1.90
18     £13.95
19      £1.10
20     £90.12
21     £64.00
22      £5.45
23      £8.00
24      £3.50
25      £3.79
26     £15.36
27      £9.49
28      £7.65
29     £12.30
30     £47.28
31      £7.65
32      £9.10
33    £117.22
34      £6.05
35     £23.00
36      £2.20
37      £4.90
38    £108.49
39     £18.00
Name: amount, dtype: str

In [42]:
def _is_date_string(s):
    # Check if the string is in the format 'dd MMM'
    if isinstance(s, str) and len(s) == 6 and s[2] == ' ':
        day = s[:2]
        month = s[3:]
        return day.isdigit() and month.isalpha()
    return False

In [26]:
df.iloc[:, 0].dtypes == str

False

In [20]:
import pymupdf

doc = pymupdf.open("C:\\Users\\willi\\Documents\\monthly_budget\\data\\raw\\tesco.pdf")

for i, page in enumerate(doc):
    tables = page.find_tables()
    n_tabs = 0
    for table in tables:
        n_tabs += 1
        # or get as Pandas DataFrame
        df = table.to_pandas()
        #print(df)

    print(f"Page {i+1} has {n_tabs} tables")

Page 1 has 1 tables
Page 2 has 2 tables
Page 3 has 0 tables
Page 4 has 0 tables


Use tabula, for this case appears to better extract